In [1]:
import pandas as pd
import numpy as np
import re

# Load your cleaned dataset
df = pd.read_csv("../data/final/medicine_final_clean.csv")

df.shape
df.head()

,url,name,composition,uses,side_effects,image_url,marketer,composition_parsed
0,https://www.1mg.com/drugs/cardiomart-h-40mg-12...,Cardiomart H 40mg/12.5mg Tablet,Telmisartan (40mg) + Hydrochlorothiazide (12.5mg),Treatment of Hypertension (high blood pressure...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Telmisartan', '40mg'), ('Hydrochlorothiazid..."
1,https://www.1mg.com/drugs/rosumart-10mg-tablet...,Rosumart 10mg Tablet,Rosuvastatin (10mg),Treatment of High cholesterol Treatment of Hig...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Rosuvastatin', '10mg')]"
2,https://www.1mg.com/drugs/tendonil-p-100mg-325...,Tendonil P 100mg/325mg Tablet,Aceclofenac (100mg) + Paracetamol (325mg),Pain relief In Pain relief Tendonil P 100mg/32...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Aceclofenac', '100mg'), ('Paracetamol', '32..."
3,https://www.1mg.com/drugs/xenoten-cv-200mg-125...,Xenoten CV 200mg/125mg Tablet,Cefpodoxime Proxetil (200mg) + Clavulanic Acid...,Treatment of Respiratory tract infection In Tr...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Proxetil', '200mg'), ('Acid', '125mg')]"
4,https://www.1mg.com/drugs/zomart-it-20mg-150mg...,Zomart IT 20mg/150mg Tablet SR,Rabeprazole (20mg) + Itopride (150mg),Treatment of Gastroesophageal reflux disease (...,Most side effects do not require any medical a...,"https://onemg.gumlet.io/q_auto,f_auto/iw8eipow...",10 Drug Mart,"[('Rabeprazole', '20mg'), ('Itopride', '150mg')]"


In [2]:
df.isna().mean().sort_values(ascending=False)

uses                  0.000101
name                  0.000004
composition           0.000004
image_url             0.000004
url                   0.000000
side_effects          0.000000
marketer              0.000000
composition_parsed    0.000000
dtype: float64

In [3]:
def normalize(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', ' ', text)  # remove special chars
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

In [4]:
df["search_text"] = (
    df["name"].fillna('') + " " +
    df["composition"].fillna('') + " " +
    df["uses"].fillna('')
)

df["search_text"] = df["search_text"].apply(normalize)

In [5]:
df[["name", "search_text"]].head(5)

,name,search_text
0,Cardiomart H 40mg/12.5mg Tablet,cardiomart h 40mg 12 5mg tablet telmisartan 40...
1,Rosumart 10mg Tablet,rosumart 10mg tablet rosuvastatin 10mg treatme...
2,Tendonil P 100mg/325mg Tablet,tendonil p 100mg 325mg tablet aceclofenac 100m...
3,Xenoten CV 200mg/125mg Tablet,xenoten cv 200mg 125mg tablet cefpodoxime prox...
4,Zomart IT 20mg/150mg Tablet SR,zomart it 20mg 150mg tablet sr rabeprazole 20m...


In [6]:
df["tokens"] = df["search_text"].apply(lambda x: set(x.split()))

In [7]:
df["tokens"].iloc[0]

{'12',
 '40mg',
 '5mg',
 'a',
 'alone',
 'and',
 'any',
 'are',
 'as',
 'at',
 'attack',
 'be',
 'benefit',
 'blood',
 'but',
 'by',
 'can',
 'cardiomart',
 'combination',
 'controlled',
 'controlling',
 'direct',
 'diuretic',
 'do',
 'easily',
 'effective',
 'effectively',
 'feel',
 'flow',
 'from',
 'generally',
 'h',
 'having',
 'heart',
 'high',
 'hydrochlorothiazide',
 'hypertension',
 'if',
 'in',
 'increases',
 'is',
 'it',
 'keep',
 'kidney',
 'known',
 'less',
 'long',
 'lower',
 'lowering',
 'medicine',
 'medicines',
 'more',
 'must',
 'namely',
 'not',
 'of',
 'one',
 'or',
 'output',
 'pill',
 'prescribed',
 'pressure',
 'problems',
 'regularly',
 'relaxing',
 'risk',
 'so',
 'stroke',
 'tablet',
 'taken',
 'taking',
 'telmisartan',
 'term',
 'the',
 'thereby',
 'this',
 'to',
 'treatment',
 'two',
 'urine',
 'used',
 'usually',
 'vessels',
 'water',
 'well',
 'when',
 'which',
 'widening',
 'will',
 'works',
 'you',
 'your'}

In [9]:
df["dosage_clean"] = df["name"].str.extract(r'(\d+\.?\d*)')
df["dosage_clean"] = df["dosage_clean"].fillna('')

In [10]:
df["dosage_all"] = df["name"].apply(
    lambda x: re.findall(r'\d+\.?\d*', str(x))
)

In [11]:
df.reset_index(inplace=True)
df.rename(columns={"index": "medicine_id"}, inplace=True)

In [13]:
df["medicine_name"] = df["name"].str.extract(r'^([A-Za-z\s]+)')
df["medicine_name"] = df["medicine_name"].str.strip().str.lower()

In [14]:
import ast

df["composition_parsed"] = df["composition_parsed"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [15]:
def extract_generic(comp_list):
    if not comp_list:
        return ""
    return " ".join([drug.lower() for drug, _ in comp_list])

df["generic_name"] = df["composition_parsed"].apply(extract_generic)

In [16]:
df[["composition_parsed", "generic_name"]].head()

,composition_parsed,generic_name
0,"[(Telmisartan, 40mg), (Hydrochlorothiazide, 12...",telmisartan hydrochlorothiazide
1,"[(Rosuvastatin, 10mg)]",rosuvastatin
2,"[(Aceclofenac, 100mg), (Paracetamol, 325mg)]",aceclofenac paracetamol
3,"[(Proxetil, 200mg), (Acid, 125mg)]",proxetil acid
4,"[(Rabeprazole, 20mg), (Itopride, 150mg)]",rabeprazole itopride


In [17]:
def parse_composition(comp):
    if pd.isna(comp):
        return []
    
    parts = comp.split('+')
    result = []
    
    for part in parts:
        part = part.strip()
        
        match = re.match(r'(.+?)\s*\(([\d\.]+mg)\)', part)
        if match:
            drug = match.group(1).strip()
            dose = match.group(2)
            result.append((drug, dose))
    
    return result

In [18]:
df["composition_parsed"] = df["composition"].apply(parse_composition)

In [19]:
def extract_generic(comp_list):
    if not comp_list:
        return ""
    return " ".join([drug.lower() for drug, _ in comp_list])

df["generic_name"] = df["composition_parsed"].apply(extract_generic)

In [21]:
df.isna().mean()
df.shape

medicine_id           0.000000
url                   0.000000
name                  0.000004
composition           0.000004
uses                  0.000101
side_effects          0.000000
image_url             0.000004
marketer              0.000000
composition_parsed    0.000000
search_text           0.000000
tokens                0.000000
dosage_clean          0.000000
dosage_all            0.000000
medicine_name         0.000944
generic_name          0.000000
dtype: float64

In [22]:
df["brand_tokens"] = df["medicine_name"].apply(
    lambda x: set(str(x).lower().split())
)

In [23]:
df["composition_tokens"] = df["generic_name"].apply(
    lambda x: set(str(x).split())
)

In [24]:
df["all_tokens"] = df.apply(
    lambda row: row["brand_tokens"] | row["composition_tokens"],
    axis=1
)

In [25]:
STOPWORDS = {"tablet", "tablets", "capsule", "capsules", "strip", "mg"}

def remove_stopwords(tokens):
    return {t for t in tokens if t not in STOPWORDS}

df["all_tokens"] = df["all_tokens"].apply(remove_stopwords)

In [26]:
df["search_priority"] = df["brand_tokens"].apply(len) + \
                        df["composition_tokens"].apply(len)

In [27]:
df[["name", "brand_tokens", "composition_tokens"]].head(10)

,name,brand_tokens,composition_tokens
0,Cardiomart H 40mg/12.5mg Tablet,"{cardiomart, h}","{hydrochlorothiazide, telmisartan}"
1,Rosumart 10mg Tablet,{rosumart},{rosuvastatin}
2,Tendonil P 100mg/325mg Tablet,"{p, tendonil}","{paracetamol, aceclofenac}"
3,Xenoten CV 200mg/125mg Tablet,"{cv, xenoten}","{cefpodoxime, clavulanic, proxetil, acid}"
4,Zomart IT 20mg/150mg Tablet SR,"{it, zomart}","{rabeprazole, itopride}"
5,Zomart LS 75mg/20mg Tablet SR,"{zomart, ls}","{rabeprazole, levosulpiride}"
6,Endomart G 2mg/500mg Tablet SR,"{g, endomart}","{metformin, glimepiride}"
7,Histadex P Syrup,"{p, histadex, syrup}",{}
8,Histadex Syrup,"{histadex, syrup}",{}
9,Tenospas 80mg/250mg Tablet,{tenospas},"{acid, drotaverine, mefenamic}"


In [29]:
df.to_csv("../data/final/medicine_ready_v2.csv", index=False)